# PyWatermark Evaluation on Colab

This notebook evaluates a trained checkpoint stored in Google Drive, shows the results directly in the notebook, and exports report-ready training graphs.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure repo and Drive root

Set `REPO_URL` once. `RUN_NAME` controls which training run is preferred for checkpoints, reports, and plots.

In [ ]:
from pathlib import Path

REPO_URL = "YOUR_REPO_URL"
PROJECT_DIR = Path('/content/PyWatermark')
DRIVE_ROOT = Path('/content/drive/MyDrive/PyWatermark')
RUN_NAME = 'robust_16'

TEST_DIR = DRIVE_ROOT / 'datasets' / 'test'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'

print('PROJECT_DIR:', PROJECT_DIR)
print('DRIVE_ROOT:', DRIVE_ROOT)
print('RUN_NAME:', RUN_NAME)
print('TEST_DIR:', TEST_DIR)
print('ARTIFACTS_DIR:', ARTIFACTS_DIR)

## 3. Clone or update the repo and install dependencies

In [ ]:
if PROJECT_DIR.exists():
    %cd /content/PyWatermark
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} PyWatermark
    %cd /content/PyWatermark

!pip install -q -r requirements.txt

## 4. Resolve the actual files to use

This cell searches Drive and prefers the selected run. If your files were saved in a slightly different folder than expected, this avoids hardcoded path mistakes.

In [ ]:
from IPython.display import display, Markdown

checkpoint_candidates = sorted(ARTIFACTS_DIR.rglob('best.pt'))
history_candidates = sorted(ARTIFACTS_DIR.rglob('metrics_history.csv'))

def pick_preferred(paths, run_name, fallback_fragment=''):
    path_strings = [str(path) for path in paths]
    for path, path_string in zip(paths, path_strings):
        if run_name and run_name in path_string:
            return path
    if fallback_fragment:
        for path, path_string in zip(paths, path_strings):
            if fallback_fragment in path_string:
                return path
    return paths[0] if paths else None

CHECKPOINT_PATH = pick_preferred(checkpoint_candidates, RUN_NAME, 'checkpoints')
HISTORY_PATH = pick_preferred(history_candidates, RUN_NAME, 'logs')

REPORT_DIR = ARTIFACTS_DIR / (f'reports_{RUN_NAME}' if RUN_NAME else 'reports')
PLOTS_DIR = ARTIFACTS_DIR / (f'plots_{RUN_NAME}' if RUN_NAME else 'plots')

display(Markdown('### Resolved Paths'))
print('TEST_DIR:', TEST_DIR)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('HISTORY_PATH:', HISTORY_PATH)
print('REPORT_DIR:', REPORT_DIR)
print('PLOTS_DIR:', PLOTS_DIR)

display(Markdown('### Checkpoint Candidates'))
for path in checkpoint_candidates:
    print(path)

display(Markdown('### History Candidates'))
for path in history_candidates:
    print(path)


## 5. Quick existence check

In [ ]:
print('test dir exists:', TEST_DIR.exists())
print('checkpoint exists:', CHECKPOINT_PATH is not None and CHECKPOINT_PATH.exists())
print('history exists:', HISTORY_PATH is not None and HISTORY_PATH.exists())

if CHECKPOINT_PATH is None:
    raise FileNotFoundError('No best.pt checkpoint was found under the artifacts directory.')
if HISTORY_PATH is None:
    print('No metrics_history.csv found. Evaluation can still run, but graph export will fail until you point HISTORY_PATH to a valid CSV.')

## 6. Evaluate the selected checkpoint

In [ ]:
%cd /content/PyWatermark
!python -m evaluation.evaluate \
    --test-dir {TEST_DIR} \
    --checkpoint {CHECKPOINT_PATH} \
    --report-dir {REPORT_DIR} \
    --batch-size 16 \
    --num-workers 2

## 7. Show the full saved report inside the notebook

In [ ]:
REPORT_PATH = REPORT_DIR / 'evaluation_report.txt'
if not REPORT_PATH.exists():
    raise FileNotFoundError(f'Report file not found: {REPORT_PATH}')

report_text = REPORT_PATH.read_text(encoding='utf-8')
print(report_text)

## 8. Show a compact results summary

In [ ]:
import re
from IPython.display import display, Markdown

def extract_metric(pattern, text, default='N/A'):
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1) if match else default

psnr = extract_metric(r'PSNR: ([0-9.]+)', report_text)
ssim = extract_metric(r'SSIM: ([0-9.]+)', report_text)
clean_acc = extract_metric(r'^clean\s+([0-9.]+)', report_text, default='N/A')
jpeg_acc = extract_metric(r'^jpeg\s+([0-9.]+)', report_text, default='N/A')
blur_acc = extract_metric(r'^blur\s+([0-9.]+)', report_text, default='N/A')
crop_acc = extract_metric(r'^crop\s+([0-9.]+)', report_text, default='N/A')
noise_acc = extract_metric(r'^noise\s+([0-9.]+)', report_text, default='N/A')
brightness_acc = extract_metric(r'^brightness\s+([0-9.]+)', report_text, default='N/A')

summary = f'''### Final Evaluation Summary

- Checkpoint: `{CHECKPOINT_PATH}`
- PSNR: `{psnr} dB`
- SSIM: `{ssim}`
- Clean bit accuracy: `{clean_acc}`
- JPEG bit accuracy: `{jpeg_acc}`
- Blur bit accuracy: `{blur_acc}`
- Crop bit accuracy: `{crop_acc}`
- Noise bit accuracy: `{noise_acc}`
- Brightness bit accuracy: `{brightness_acc}`
'''

display(Markdown(summary))

## 9. Export training graphs for the report

In [ ]:
if HISTORY_PATH is None or not HISTORY_PATH.exists():
    raise FileNotFoundError(f'History file not found: {HISTORY_PATH}')

%cd /content/PyWatermark
!python -m evaluation.plot_training_curves \
    --history {HISTORY_PATH} \
    --output-dir {PLOTS_DIR} \
    --title-prefix "PyWatermark {RUN_NAME}"

## 10. List generated outputs

In [ ]:
!ls {REPORT_DIR}
!ls {PLOTS_DIR}